# 04 · Clustering, segment profiling & recommendations
*Sections 11–16.*

In [1]:
import sys, warnings; sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go
from src import cleaning as cl, features as ft, segmentation as sg, viz
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 40
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram
df = pd.read_parquet("../data/processed/lines_clean.parquet")
c = ft.score_rfm(ft.customer_table(df))
print(len(c), "customers")

4320 customers


## 11 · Preprocessing for clustering
1. **Log transform** (`log1p`) to compress the wholesale skew.
2. **StandardScaler** → mean 0, variance 1, otherwise £ dwarfs days in Euclidean distance.
3. `CustomerID` is **not** a feature.

Skewness before / after:

In [2]:
raw_feats = c[sg.FEATURES]
print("skew BEFORE:", raw_feats.skew().round(2).to_dict())
print("skew AFTER :", np.log1p(raw_feats).skew().round(2).to_dict())
Xs, scaler = sg.preprocess(c)
print("scaled mean:", Xs.mean(0).round(3), "std:", Xs.std(0).round(3))

skew BEFORE: {'Recency': 1.25, 'Frequency': 11.94, 'Monetary': 21.54}
skew AFTER : {'Recency': -0.32, 'Frequency': 1.21, 'Monetary': 0.37}
scaled mean: [0. 0. 0.] std: [1. 1. 1.]


In [3]:
fig = go.Figure()
for f in sg.FEATURES:
    fig.add_histogram(x=np.log1p(c[f]), name=f"log {f}", opacity=.6, nbinsx=50)
fig.update_layout(barmode="overlay", title="After log1p the three distributions are usable")
viz._style(fig, "After log1p the three distributions are usable", 340)

## 12 · Clustering & evaluation
We don't pick K because a curve "looks right". Four independent criteria are used: **WCSS elbow, silhouette, Davies-Bouldin and bootstrap stability** (ARI between a K-Means fit on all data and fits on random 80% subsets), plus **actionability**.

In [4]:
ev = sg.evaluate_k(Xs)
display(ev.round(3))
viz.elbow_silhouette(ev)

,k,WCSS,Silhouette,DaviesBouldin,WCSS_drop,Kink
0,2,"6,446.65",0.43,0.89,NaN,NaN
1,3,"4,827.12",0.34,1.04,"1,619.53",NaN
2,4,"3,896.53",0.34,1.02,930.59,688.94
3,5,"3,245.79",0.32,0.98,650.74,279.85
4,6,"2,819.56",0.31,1.02,426.23,224.52
5,7,"2,511.95",0.30,0.99,307.61,118.61
6,8,"2,309.35",0.30,1.01,202.60,105.02
7,9,"2,129.74",0.28,1.03,179.61,22.99
8,10,"1,971.82",0.28,1.02,157.92,21.69


In [5]:
stab = pd.DataFrame([(k, *sg.stability(Xs, k, n_boot=10)) for k in (2,3,4,5,6)], columns=["k","ARI_mean","ARI_std"]).round(3)
stab

,k,ARI_mean,ARI_std
0,2,0.99,0.01
1,3,0.97,0.04
2,4,0.94,0.04
3,5,0.92,0.05
4,6,0.88,0.04


**Choosing K = 4.**
* **k=2** has the best silhouette (0.43) but only separates "active vs inactive" — it cannot drive different marketing actions.
* The elbow *kink* (largest second difference) is at k=4; WCSS flattens thereafter.
* Silhouette is practically identical for k=3 and k=4 (≈0.34) → **no separation is lost by going to 4**, and Davies-Bouldin is similar.
* **Stability:** ARI falls steadily as K grows (k=3 ≈ 0.97, **k=4 ≈ 0.94**, k=5 ≈ 0.92, k=6 ≈ 0.88). 0.94 is still very reproducible, and every extra cluster beyond 4 buys less separation and less stability.
* **Actionability:** 4 clusters map to four distinct playbooks (below).

Silhouette of 0.34 is *moderate* — customer behaviour is a continuum, not islands. We state that rather than overclaim.

### Algorithm comparison (guideline table)
**Hierarchical (Ward)** on a 1,500-customer sample: computationally heavy on all data; the dendrogram lets us verify that the tree really splits into ~4 big branches.

In [6]:
import plotly.figure_factory as ff
idx = np.random.default_rng(42).choice(len(Xs), 1500, replace=False)
dend = ff.create_dendrogram(Xs[idx], orientation="bottom", linkagefun=lambda x: linkage(x, "ward"),
                            color_threshold=18)
dend.update_xaxes(showticklabels=False)
viz._style(dend, "Ward dendrogram (1,500-customer sample)", 420)

In [7]:
hc = AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(Xs[idx])
km_all = KMeans(4, n_init=20, random_state=42).fit(Xs)
print("ARI hierarchical vs K-Means on same sample:", round(adjusted_rand_score(hc, km_all.labels_[idx]), 3))

ARI hierarchical vs K-Means on same sample: 0.41


Agreement is **moderate (ARI ≈ 0.4)**, which is what we should expect: both methods minimise variance but cut a *continuous* cloud at different places. What they share is the broad structure — a large lapsed group, a small high-frequency core and an in-between recent group — so the 4-group picture is not a K-Means artefact, but the exact borders are soft.

**DBSCAN** — used as the guideline suggests: not to segment, but to **isolate the whales / anomalies**.

In [8]:
lab = sg.dbscan_outliers(Xs, eps=0.4, min_samples=10)
c["DBSCAN_Outlier"] = lab == -1
print(f"DBSCAN noise points: {c.DBSCAN_Outlier.sum()} customers ({c.DBSCAN_Outlier.mean():.1%}) holding {c[c.DBSCAN_Outlier].Monetary.sum()/c.Monetary.sum():.1%} of revenue")
c.groupby("DBSCAN_Outlier")[["Recency","Frequency","Monetary","AvgBasketUnits"]].median().round(1)

DBSCAN noise points: 162 customers (3.8%) holding 37.2% of revenue


,Recency,Frequency,Monetary,AvgBasketUnits
DBSCAN_Outlier,,,,
False,52.00,2.00,633.70,160.00
True,16.50,9.00,"7,011.20",302.60


3.8% of customers (162) are structurally unusual yet hold **37% of net revenue** — far-out bulk buyers (median basket ~300 units, median net spend ~£7.0k) who also order recently (median 16 days). These **whales are real value but would distort K-Means centroids**, so they are flagged (`DBSCAN_Outlier`, `Whale` = top 1% by revenue) and handled with account-management rather than mass marketing.

In [9]:
km = KMeans(4, n_init=20, random_state=42).fit(Xs)
c["Cluster"] = km.labels_
prof = sg.profile(c, "Cluster")
names = sg.name_clusters(prof)
c["ClusterName"] = c.Cluster.map(names)
prof = sg.profile(c, "ClusterName")
prof.sort_values("Revenue_share_%", ascending=False)

,Customers,Recency_median,Frequency_median,Monetary_median,Monetary_sum,AOV_median,BasketUnits_median,ReturnRate_mean,UK_share,Customer_share_%,Revenue_share_%
ClusterName,,,,,,,,,,,
Power Core,706,9.00,10.00,"3,539.82","5,324,092.35",361.39,211.35,0.02,0.89,16.34,64.38
Slipping Regulars,1186,52.00,4.00,"1,314.06","1,985,074.10",351.56,206.00,0.02,0.89,27.45,24.00
Lapsed One-Timers,1542,187.00,1.00,294.18,"542,514.36",226.84,125.00,0.02,0.91,35.69,6.56
Fresh Explorers,886,20.00,2.00,400.02,"417,956.51",221.62,136.83,0.01,0.91,20.51,5.05


## 13 · Segment profiling

In [10]:
viz.radar(c)

In [11]:
viz.segment_bubbles(prof)

In [12]:
viz.galaxy(c.assign(zR=Xs[:,0], zF=Xs[:,1], zM=Xs[:,2]))

**Names come from the statistics, not a template:**
| Cluster | Evidence |
|---|---|
| **Power Core** | ~16% of customers, **~64% of revenue**, median 10 orders, last purchase ~9 days ago, median spend £3.5k |
| **Slipping Regulars** | median 4 orders and £1.3k, but **last purchase ~52 days ago** — ~27% of customers and ~24% of revenue; the early-churn zone |
| **Fresh Explorers** | recent (≈20 days), 2 orders, £400 median — the warm second-purchase pool |
| **Lapsed One-Timers** | one order, last seen ~6 months ago (187 days), 36% of customers but **6.6% of revenue** |

Return rates and UK share are almost identical across clusters (~2%, ~90%) → **geography and returns do not explain the segments; frequency and recency do.** That is a finding, not a gap.

### Do the two segmentations agree? (rule-based RFM vs data-driven clusters)

In [13]:
ct = pd.crosstab(c.RFM_Segment, c.ClusterName)
print("ARI(RFM rules, K-Means):", round(adjusted_rand_score(c.RFM_Segment, c.ClusterName), 3))
ct

ARI(RFM rules, K-Means): 0.421


ClusterName,Fresh Explorers,Lapsed One-Timers,Power Core,Slipping Regulars
RFM_Segment,,,,
At-Risk High-Value,0,118,5,336
Champions,10,0,670,260
Hibernating / Lost,0,1037,0,35
Loyal Customers,444,28,31,493
Needs Attention,133,359,0,52
New Inflow,299,0,0,10


In [14]:
viz.sankey(c)

Moderate agreement (ARI ≈ 0.4): rules and K-Means find the same *story* but cut it in different places. Notably the rule-based **Champions** split between *Power Core* and *Slipping Regulars* — K-Means uses raw log distance, so a frequent-but-quieter customer lands in "Slipping" even though the rules still call them Champions (R=4). **Use K-Means for marketing sizing, RFM rules for customer-level triggers** — and treat borderline customers as watch-list.

## 14 · Business recommendations
| Segment | Size / value | Action | KPI |
|---|---|---|---|
| **Power Core** (≈16% of customers, ≈64% revenue) | protect | Dedicated account management for the top 1% whales; early access to Q4 range in **August**; no blanket discounts | retention of top-100, order frequency |
| **Slipping Regulars** (≈27% of customers, ≈24% of revenue) | **highest ROI** | Trigger a win-back when a regular passes ~45 days without ordering (their typical rhythm is shorter); bundle/volume-tier offers | % reactivated within 30 days |
| **Fresh Explorers** | grow | Welcome → 2nd-order sequence within 3 weeks; cross-sell from first basket | 2nd-order rate |
| **Lapsed One-Timers** | cheap tail | One automated reactivation e-mail, then suppress; do not spend paid media | cost per reactivation |
| **Guest checkouts** (15% of revenue) | blind spot | Incentivise account creation at checkout so they become scoreable | % identified orders |
| **Operations** | | Pre-build stock from September; maintenance on Saturday/evenings; review the 22 dead-stock SKUs; audit the few accounts behind most returns | |

## 15 · Limitations & next steps
* **One year of data** (and a partial December): seasonality is a single observation; can't separate trend from season.
* **Descriptive, not causal.** We can say Champions buy often; not that a perk *makes* them buy.
* **Customer 16446-type events** (huge order + full cancellation) are rare but swing totals; handled via net revenue and outlier flags.
* **25% of lines are anonymous**, so customer-level numbers describe the *identifiable* ~85% of product revenue.
* Silhouette 0.34 = overlapping, continuous segments; the labels are tools, not natural kinds.
* **Next:** (1) CLV model (BG/NBD + Gamma-Gamma) to replace the recency proxy; (2) A/B test the win-back trigger; (3) product-affinity (market-basket) per segment; (4) retrain monthly with a rolling snapshot.

## 16 · Final conclusion
The business is a **trade-led giftware wholesaler with a seasonal Q4 peak**. A small **Power Core (~16% of customers) delivers ~64% of revenue**; the largest *recoverable* pool is the **Slipping Regulars (≈£2.0M, ≈24%)**, and the large Lapsed One-Timer group is cheap noise. Cleaning decisions (net-of-returns, guest handling, de-duplication) matter: they remove phantom bestsellers and give trustworthy numbers. RFM rules give transparent, customer-level triggers, K-Means gives a stable (bootstrap ARI≈0.94) four-way market map, and DBSCAN protects both from whale distortion.

In [15]:
c.to_parquet("../data/processed/customers_notebook.parquet")
print("done")

done
